In [1]:
# ============================================================
# SMART WATER AI
# Water Demand Prediction + Anomaly Detection + NLP
# ============================================================

# Install if required:
# !pip install -U gradio pandas numpy scikit-learn joblib

import gradio as gr
import pandas as pd
import numpy as np
import joblib
import re
import warnings

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")


# ============================================================
# 1. LOAD TRAINED WATER ML MODELS
# ============================================================

try:
    loaded_model = joblib.load(
        "smart_water_demand_model.pkl"
    )

    loaded_anomaly_model = joblib.load(
        "water_anomaly_model.pkl"
    )

    loaded_anomaly_scaler = joblib.load(
        "water_anomaly_scaler.pkl"
    )

    models_loaded = True

except Exception as e:

    print("Model loading error:", e)
    models_loaded = False


# ============================================================
# 2. NLP TRAINING DATA
# ============================================================

complaint_data = [

    # -------------------------
    # Leakage
    # -------------------------

    ("There is a water leakage in our street", "Leakage"),
    ("Water is leaking from the pipeline", "Leakage"),
    ("Pipeline is broken and water is flowing", "Leakage"),
    ("There is leakage near my house", "Leakage"),
    ("Water pipe is damaged", "Leakage"),
    ("Large amount of water is leaking", "Leakage"),
    ("Water is continuously leaking", "Leakage"),
    ("പൈപ്പിൽ വെള്ളം ചോർച്ചയുണ്ട്", "Leakage"),
    ("വാട്ടർ പൈപ്പ് പൊട്ടി വെള്ളം പോകുന്നു", "Leakage"),

    # -------------------------
    # Low Water Supply
    # -------------------------

    ("Water supply is very low", "Low Water Supply"),
    ("We are not getting enough water", "Low Water Supply"),
    ("Water pressure is very low", "Low Water Supply"),
    ("There is no proper water supply", "Low Water Supply"),
    ("Water supply stopped in our area", "Low Water Supply"),
    ("Very little water is coming", "Low Water Supply"),
    ("No water is available today", "Low Water Supply"),
    ("വെള്ളം കുറവാണ്", "Low Water Supply"),
    ("വെള്ളത്തിന്റെ pressure കുറവാണ്", "Low Water Supply"),

    # -------------------------
    # High Consumption
    # -------------------------

    ("Water consumption is very high", "High Consumption"),
    ("Our water usage has suddenly increased", "High Consumption"),
    ("Water usage is more than normal", "High Consumption"),
    ("Consumption increased suddenly", "High Consumption"),
    ("We are using too much water", "High Consumption"),
    ("Daily water consumption is very high", "High Consumption"),
    ("Water demand is increasing", "High Consumption"),
    ("വെള്ളത്തിന്റെ ഉപയോഗം വളരെ കൂടുതലാണ്", "High Consumption"),
    ("water consumption പെട്ടെന്ന് കൂടി", "High Consumption"),

    # -------------------------
    # Water Quality
    # -------------------------

    ("Water quality is bad", "Water Quality"),
    ("The water is dirty", "Water Quality"),
    ("Water has a bad smell", "Water Quality"),
    ("There is mud in the water", "Water Quality"),
    ("The supplied water is not clean", "Water Quality"),
    ("Water looks contaminated", "Water Quality"),
    ("We are getting dirty water", "Water Quality"),
    ("വെള്ളം മലിനമാണ്", "Water Quality"),
    ("വെള്ളത്തിന് ദുർഗന്ധമുണ്ട്", "Water Quality"),

    # -------------------------
    # Meter Problem
    # -------------------------

    ("Water meter is not working", "Meter Problem"),
    ("The water meter is damaged", "Meter Problem"),
    ("Meter reading is incorrect", "Meter Problem"),
    ("Water meter shows wrong value", "Meter Problem"),
    ("Our meter stopped working", "Meter Problem"),
    ("Meter is giving abnormal readings", "Meter Problem"),
    ("There is a problem with the water meter", "Meter Problem"),
    ("water meter work ചെയ്യുന്നില്ല", "Meter Problem"),
    ("meter reading തെറ്റാണ്", "Meter Problem"),

    # -------------------------
    # Other
    # -------------------------

    ("I have a question about water supply", "Other"),
    ("I need information about water service", "Other"),
    ("Please provide water supply information", "Other"),
    ("I want to know about the water service", "Other"),
    ("How can I report a water issue", "Other"),
]


# ============================================================
# 3. CREATE NLP DATAFRAME
# ============================================================

nlp_df = pd.DataFrame(
    complaint_data,
    columns=["text", "category"]
)


# ============================================================
# 4. TEXT CLEANING FUNCTION
# ============================================================

def clean_text(text):

    text = str(text).lower()

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    # Remove unnecessary punctuation
    text = re.sub(r"[^\w\s]", "", text)

    return text.strip()


nlp_df["clean_text"] = nlp_df["text"].apply(clean_text)


# ============================================================
# 5. TF-IDF
# ============================================================

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=1000
)

X_nlp = tfidf_vectorizer.fit_transform(
    nlp_df["clean_text"]
)

y_nlp = nlp_df["category"]


# ============================================================
# 6. NLP CLASSIFIER
# ============================================================

nlp_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

nlp_model.fit(
    X_nlp,
    y_nlp
)


# ============================================================
# 7. NLP PREDICTION FUNCTION
# ============================================================

def analyze_complaint(complaint):

    if complaint is None or complaint.strip() == "":
        return (
            "⚠️ Please enter a water-related complaint.",
            "",
            ""
        )

    cleaned = clean_text(complaint)

    vectorized_text = tfidf_vectorizer.transform(
        [cleaned]
    )

    prediction = nlp_model.predict(
        vectorized_text
    )[0]

    probabilities = nlp_model.predict_proba(
        vectorized_text
    )[0]

    confidence = np.max(probabilities) * 100

    return (
        f"### 📝 Complaint Category\n"
        f"**{prediction}**",

        f"### 🎯 Confidence\n"
        f"**{confidence:.2f}%**",

        f"### 🔍 Processed Text\n"
        f"`{cleaned}`"
    )


# ============================================================
# 8. WATER DEMAND PREDICTION FUNCTION
# ============================================================

def predict_water_demand(
    area,
    connections,
    temperature,
    rainfall,
    day_of_week,
    month,
    previous_consumption,
    average_7,
    average_30,
    available_supply
):

    if not models_loaded:

        return (
            "❌ Models not loaded. Check your .pkl files.",
            0,
            0,
            "ERROR",
            "❌ Model Error"
        )

    try:

        # --------------------------------------
        # Create input dataframe
        # --------------------------------------

        new_data = pd.DataFrame({

            "Area": [area],

            "Number_of_Connections": [
                connections
            ],

            "Temperature_C": [
                temperature
            ],

            "Rainfall_mm": [
                rainfall
            ],

            "Day_of_Week": [
                day_of_week
            ],

            "Month": [
                month
            ],

            "Previous_Day_Consumption_L": [
                previous_consumption
            ],

            "7_Day_Average_Consumption_L": [
                average_7
            ],

            "30_Day_Average_Consumption_L": [
                average_30
            ],

            "Available_Water_Supply_L": [
                available_supply
            ]
        })


        # --------------------------------------
        # Water demand prediction
        # --------------------------------------

        prediction = loaded_model.predict(
            new_data
        )

        predicted_consumption = float(
            prediction[0]
        )


        # --------------------------------------
        # Demand ratio
        # --------------------------------------

        if available_supply <= 0:

            ratio = 0

        else:

            ratio = (
                predicted_consumption
                / available_supply
            )


        ratio_percentage = ratio * 100


        # --------------------------------------
        # Risk classification
        # --------------------------------------

        if ratio >= 1.15:

            risk = "HIGH"

            risk_message = (
                "🔴 HIGH DEMAND RISK"
            )

        elif ratio >= 0.95:

            risk = "MEDIUM"

            risk_message = (
                "🟠 MEDIUM DEMAND RISK"
            )

        else:

            risk = "LOW"

            risk_message = (
                "🟢 LOW DEMAND RISK"
            )


        # --------------------------------------
        # Anomaly detection
        # --------------------------------------

        anomaly_input = pd.DataFrame({

            "Water_Consumption_L": [
                predicted_consumption
            ],

            "Previous_Day_Consumption_L": [
                previous_consumption
            ],

            "7_Day_Average_Consumption_L": [
                average_7
            ],

            "30_Day_Average_Consumption_L": [
                average_30
            ],

            "Demand_Ratio": [
                ratio
            ]

        })


        anomaly_scaled = (
            loaded_anomaly_scaler.transform(
                anomaly_input
            )
        )


        anomaly_result = (
            loaded_anomaly_model.predict(
                anomaly_scaled
            )
        )


        if anomaly_result[0] == -1:

            anomaly_status = (
                "⚠️ ABNORMAL "
                "WATER CONSUMPTION DETECTED"
            )

        else:

            anomaly_status = (
                "✅ NORMAL WATER CONSUMPTION"
            )


        # --------------------------------------
        # Final result
        # --------------------------------------

        result = f"""
# 💧 Water Demand Prediction

### 📍 Area
**{area}**

### 🔮 Predicted Consumption
**{predicted_consumption:,.2f} L**

### 📊 Demand Ratio
**{ratio_percentage:.2f}%**

### ⚠️ Demand Risk
**{risk_message}**

### 🚨 Consumption Status
**{anomaly_status}**
"""

        return (
            result,
            round(predicted_consumption, 2),
            round(ratio_percentage, 2),
            risk,
            anomaly_status
        )

    except Exception as e:

        return (
            f"❌ Error: {str(e)}",
            0,
            0,
            "ERROR",
            "❌ Prediction Failed"
        )


# ============================================================
# 9. CLEAR FUNCTION
# ============================================================

def clear_all():

    return (
        "Ward_01",
        500,
        28,
        0,
        0,
        1,
        40000,
        40000,
        40000,
        50000,
        "",
        0,
        0,
        "LOW",
        "Waiting..."
    )


# ============================================================
# 10. CUSTOM CSS
# ============================================================

custom_css = """

body {
    background:
        radial-gradient(
            circle at top left,
            #123b4a,
            #07131f 55%,
            #02070d
        );
}

.gradio-container {
    max-width: 1250px !important;
    margin: auto !important;
}

#main-title {
    text-align: center;
    padding: 25px;
    border-radius: 20px;
    background: linear-gradient(
        135deg,
        #0e3848,
        #102b3b
    );
    border: 1px solid #23c4b4;
    box-shadow:
        0 10px 40px rgba(0,0,0,.35);
}

#main-title h1 {
    color: #ffffff;
    font-size: 38px;
}

#main-title p {
    color: #23c4b4;
    font-size: 17px;
}

.card {
    border-radius: 18px !important;
    border: 1px solid #23495a !important;
    background: rgba(
        15,
        35,
        48,
        0.85
    ) !important;
    padding: 15px !important;
}

button {
    border-radius: 12px !important;
}

#predict-btn {
    background: linear-gradient(
        90deg,
        #12a89d,
        #23c4b4
    ) !important;

    color: white !important;
    font-weight: bold !important;
}

#nlp-btn {
    background: linear-gradient(
        90deg,
        #6c5ce7,
        #8e7dff
    ) !important;

    color: white !important;
    font-weight: bold !important;
}

#result {
    border-radius: 18px !important;
    padding: 20px !important;
}

footer {
    display: none !important;
}

"""


# ============================================================
# 11. GRADIO INTERFACE
# ============================================================

with gr.Blocks(
    theme=gr.themes.Soft(),
    css=custom_css,
    title="Smart Water AI"
) as demo:

    # --------------------------------------
    # Header
    # --------------------------------------

    gr.HTML(
        """
        <div id="main-title">

            <h1>
                💧 Smart Water AI
            </h1>

            <p>
                Water Demand Prediction •
                Anomaly Detection •
                NLP Complaint Analysis
            </p>

        </div>
        """
    )


    # ========================================================
    # TABS
    # ========================================================

    with gr.Tabs():

        # ====================================================
        # TAB 1 — WATER DEMAND
        # ====================================================

        with gr.Tab("💧 Water Demand Prediction"):

            gr.Markdown(
                """
                ## 🔮 Predict Future Water Demand

                Enter the current water and environmental
                information to predict expected consumption.
                """
            )

            with gr.Row():

                with gr.Column(
                    scale=1,
                    elem_classes="card"
                ):

                    area = gr.Dropdown(
                        choices=[
                            f"Ward_{i:02d}"
                            for i in range(1, 21)
                        ],
                        value="Ward_01",
                        label="📍 Area"
                    )

                    connections = gr.Number(
                        value=500,
                        label="🏠 Number of Connections"
                    )

                    temperature = gr.Number(
                        value=28,
                        label="🌡️ Temperature (°C)"
                    )

                    rainfall = gr.Number(
                        value=0,
                        label="🌧️ Rainfall (mm)"
                    )

                    day_of_week = gr.Slider(
                        minimum=0,
                        maximum=6,
                        step=1,
                        value=0,
                        label="📅 Day of Week"
                    )

                    month = gr.Slider(
                        minimum=1,
                        maximum=12,
                        step=1,
                        value=1,
                        label="🗓️ Month"
                    )


                with gr.Column(
                    scale=1,
                    elem_classes="card"
                ):

                    previous_consumption = gr.Number(
                        value=40000,
                        label="💧 Previous Day Consumption (L)"
                    )

                    average_7 = gr.Number(
                        value=40000,
                        label="📊 7-Day Average Consumption (L)"
                    )

                    average_30 = gr.Number(
                        value=40000,
                        label="📈 30-Day Average Consumption (L)"
                    )

                    available_supply = gr.Number(
                        value=50000,
                        label="🚰 Available Water Supply (L)"
                    )


            with gr.Row():

                predict_btn = gr.Button(
                    "🔮 Predict Water Demand",
                    variant="primary",
                    elem_id="predict-btn"
                )

                clear_btn = gr.Button(
                    "🧹 Clear"
                )


            result = gr.Markdown(
                label="Prediction Result",
                elem_id="result"
            )


            with gr.Row():

                predicted_output = gr.Number(
                    label="💧 Predicted Consumption (L)"
                )

                ratio_output = gr.Number(
                    label="📊 Demand Ratio (%)"
                )

                risk_output = gr.Textbox(
                    label="⚠️ Risk Level"
                )

                anomaly_output = gr.Textbox(
                    label="🚨 Anomaly Status"
                )


            predict_btn.click(

                fn=predict_water_demand,

                inputs=[
                    area,
                    connections,
                    temperature,
                    rainfall,
                    day_of_week,
                    month,
                    previous_consumption,
                    average_7,
                    average_30,
                    available_supply
                ],

                outputs=[
                    result,
                    predicted_output,
                    ratio_output,
                    risk_output,
                    anomaly_output
                ]
            )


        # ====================================================
        # TAB 2 — NLP
        # ====================================================

        with gr.Tab("📝 NLP Water Complaint Analysis"):

            gr.Markdown(
                """
                ## 🧠 AI Water Complaint Analyzer

                Enter a water-related complaint or message.
                The NLP model will automatically classify it.
                """
            )


            with gr.Row():

                with gr.Column(
                    scale=1,
                    elem_classes="card"
                ):

                    complaint_input = gr.Textbox(
                        label="📝 Enter Water Complaint",
                        placeholder=(
                            "Example: "
                            "There is a water leakage "
                            "near our house"
                        ),
                        lines=6
                    )


                    nlp_btn = gr.Button(
                        "🧠 Analyze Complaint",
                        variant="primary",
                        elem_id="nlp-btn"
                    )


                with gr.Column(
                    scale=1,
                    elem_classes="card"
                ):

                    complaint_category = gr.Markdown(
                        "### 📝 Complaint Category\nWaiting..."
                    )

                    complaint_confidence = gr.Markdown(
                        "### 🎯 Confidence\nWaiting..."
                    )

                    processed_text = gr.Markdown(
                        "### 🔍 Processed Text\nWaiting..."
                    )


            nlp_btn.click(

                fn=analyze_complaint,

                inputs=[
                    complaint_input
                ],

                outputs=[
                    complaint_category,
                    complaint_confidence,
                    processed_text
                ]
            )


            gr.Examples(

                examples=[
                    [
                        "There is a water leakage "
                        "from the pipeline"
                    ],

                    [
                        "We are not getting "
                        "enough water"
                    ],

                    [
                        "Water consumption "
                        "is very high"
                    ],

                    [
                        "The supplied water "
                        "is dirty"
                    ],

                    [
                        "Water meter is "
                        "not working"
                    ]
                ],

                inputs=complaint_input
            )


        # ====================================================
        # TAB 3 — ABOUT PROJECT
        # ====================================================

        with gr.Tab("ℹ️ About Project"):

            gr.Markdown(
                """
                # 💧 Smart Water AI

                ### Machine Learning Modules

                **1. Water Demand Prediction**

                XGBoost / Random Forest regression is used
                to predict future water consumption.

                **2. Abnormal Consumption Detection**

                Isolation Forest identifies unusual
                consumption patterns.

                **3. NLP Complaint Analysis**

                TF-IDF converts complaint text into numerical
                features and Logistic Regression classifies
                the complaint.

                ### NLP Categories

                - 💧 Leakage
                - 🚰 Low Water Supply
                - 📈 High Consumption
                - 🧪 Water Quality
                - 🔧 Meter Problem
                - ❓ Other

                ### Technologies

                Python • Pandas • NumPy • Scikit-learn
                • XGBoost • Joblib • Gradio • NLP
                """
            )


    # ========================================================
    # CLEAR BUTTON
    # ========================================================

    clear_btn.click(

        fn=clear_all,

        inputs=[],

        outputs=[
            area,
            connections,
            temperature,
            rainfall,
            day_of_week,
            month,
            previous_consumption,
            average_7,
            average_30,
            available_supply,
            result,
            predicted_output,
            ratio_output,
            risk_output,
            anomaly_output
        ]
    )


# ============================================================
# 12. LAUNCH
# ============================================================

demo.launch(
    share=True,
    inbrowser=True
)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/09/28 10:37:51 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: i/o timeout
